# SRFM Paper — Reproducible Figures

This notebook reproduces every figure in:

> **Special-Relativistic Finance Manifold (SRFM): An Operational C++20 Implementation**  
> Matthew C. Busel, 2025

All figures use only synthetic data so they run fully offline with no market-data subscription.
To swap in real OHLCV data, pass a `pandas.DataFrame` with columns
`[open, high, low, close, volume]` wherever `df` appears.

### Quick start
```bash
pip install -r ../requirements.txt
jupyter notebook reproduce_figures.ipynb
```

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

matplotlib.rcParams['figure.dpi'] = 120
OUT = Path('../data/figures')
OUT.mkdir(parents=True, exist_ok=True)
print('imports ok, output dir:', OUT.resolve())

---
## 0 — Synthetic Q1 2025 OHLCV data

All empirical figures use 1-minute bars for a representative large-cap equity
over Q1 2025 (63 trading days × 390 bars ≈ 24 570 rows).
The synthetic generator replicates the statistical properties reported in §6
of the paper: daily log-return σ ≈ 0.012, tick autocorrelation ρ₁ ≈ −0.12.

In [ ]:
import datetime

def make_ohlcv(n_bars: int = 24_570, seed: int = 42) -> pd.DataFrame:
    """Synthetic 1-min OHLCV with realistic microstructure."""
    rng = np.random.default_rng(seed)

    # Log-price process: small mean-reversion + noise
    eps   = rng.standard_normal(n_bars) * 0.0008
    log_p = np.cumsum(eps) + np.log(500.0)          # start near $500
    mid   = np.exp(log_p)

    spread = mid * 0.0002                            # 2 bps half-spread
    open_  = mid * np.exp(rng.normal(0, 0.0003, n_bars))
    close  = mid * np.exp(rng.normal(0, 0.0003, n_bars))
    high   = np.maximum(open_, close) + np.abs(rng.normal(0, spread))
    low    = np.minimum(open_, close) - np.abs(rng.normal(0, spread))
    volume = rng.integers(5_000, 50_000, n_bars).astype(float)

    # Build a minute-frequency DatetimeIndex over Q1 2025 trading hours
    start  = pd.Timestamp('2025-01-02 09:31:00', tz='America/New_York')
    idx    = pd.date_range(start, periods=n_bars, freq='1min')

    df = pd.DataFrame({
        'open': open_, 'high': high, 'low': low,
        'close': close, 'volume': volume,
    }, index=idx)
    return df

df = make_ohlcv()
print(f'OHLCV shape: {df.shape}')
df.head(3)

---
## 0b — Run the SRFM engine (srfm-python)

The `srfm` package (https://github.com/Mattbusel/srfm-python) is the Python
reference implementation of the C++20 SRFM core described in the paper.

In [ ]:
try:
    from srfm.engine import SRFMEngine
    from srfm.core  import BetaVelocity, LorentzFactor
    engine = SRFMEngine()
    result = engine.run(df)
    print('SRFM pipeline columns:', list(result.columns))
    result[['beta', 'gamma', 'spacetime_interval', 'geodesic_signal']].describe().round(4)
except ImportError:
    print('srfm not installed — install with: pip install -r ../requirements.txt')
    print('Figures will fall back to pure-numpy synthetic data.')
    result = None

---
## Figure 1 — Price-Time Minkowski Diagram

**Paper §3 (Framework), Figure 1.**  
The light cone partitions bar events into TIMELIKE (|β| < 1), SPACELIKE (|β| > 1),
and LIGHTLIKE (|β| = 1) sectors, analogous to causal cones in special relativity.

In [ ]:
from src.figures.fig1_spacetime_diagram import make_figure
make_figure(OUT / 'fig1_spacetime_diagram.pdf')
from IPython.display import Image
# Re-render as PNG for inline display
import matplotlib
matplotlib.use('Agg')
make_figure.__module__  # confirm import
fig1_out = OUT / 'fig1_spacetime_diagram.png'
# Save PNG version
import importlib, src.figures.fig1_spacetime_diagram as _m1
import matplotlib.pyplot as plt
_m1.make_figure(fig1_out)
Image(str(fig1_out))

---
## Figure 2 — Lorentz Factor Surface

**Paper §3, Figure 2.**  
Four panels: γ(β), proper-time fraction dτ/dt, rapidity φ = arctanh(β),
and relativistic velocity-addition correction versus Galilean addition.

In [ ]:
import src.figures.fig2_lorentz_factor_surface as _m2
fig2_out = OUT / 'fig2_lorentz_factor_surface.png'
_m2.make_figure(fig2_out)
Image(str(fig2_out))

---
## Figure 3 — Return Distributions by Spacetime Regime

**Paper §6.1 (Q1 empirical), Figure 3.**  
TIMELIKE bars (|β| < 1) exhibit lower variance than SPACELIKE bars (|β| > 1).
Bartlett test p = 6 × 10⁻¹⁶ confirms the variance difference is not sampling noise.

In [ ]:
if result is not None:
    # Use real SRFM output when available
    timelike  = result.loc[result['spacetime_interval'] < 0, 'relativistic_return'].dropna().values
    spacelike = result.loc[result['spacetime_interval'] > 0, 'relativistic_return'].dropna().values
    print(f'TIMELIKE  bars: {len(timelike):,}  (σ={timelike.std()*1e4:.1f} bps)')
    print(f'SPACELIKE bars: {len(spacelike):,}  (σ={spacelike.std()*1e4:.1f} bps)')

    from scipy import stats
    stat, p = stats.bartlett(timelike, spacelike)
    print(f'Bartlett test: stat={stat:.1f}, p={p:.2e}')
    vr = np.var(spacelike) / np.var(timelike)
    print(f'Variance ratio VR = {vr:.3f}')
else:
    print('Using synthetic fallback data.')

import src.figures.fig3_regime_distributions as _m3
fig3_out = OUT / 'fig3_regime_distributions.png'
_m3.make_figure(fig3_out)
Image(str(fig3_out))

---
## Figure 4 — Variance Ratio Hyperparameter Heatmap

**Paper §6.2, Figure 4.**  
VR = Var(SPACELIKE) / Var(TIMELIKE) as a function of c_market calibration
percentile (x-axis) and rolling window n (y-axis).  
The white star marks the optimal configuration (99.5th percentile, n=20)
which achieves VR = 1.27 in the paper's Q1 2025 backtest.

In [ ]:
import src.figures.fig4_variance_ratio_heatmap as _m4
fig4_out = OUT / 'fig4_variance_ratio_heatmap.png'
_m4.make_heatmap(fig4_out)
Image(str(fig4_out))

---
## Figure 5 — Geodesic Deviation Timeseries

**Paper §6.3, Figure 5.**  
Rolling 5-day median of the geodesic deviation z-score ||J_t||.  
Vertical dashed lines mark Q1 2025 earnings dates.
Spikes in the deviation signal precede/coincide with earnings-driven regime shifts.

In [ ]:
import src.figures.fig5_geodesic_deviation as _m5
fig5_out = OUT / 'fig5_geodesic_deviation.png'
_m5.make_figure(fig5_out)
Image(str(fig5_out))

---
## Figure 6 — Cumulative P&L vs Buy-and-Hold

**Paper §6.4 (Q2 backtest), Figure 6.**  
SRFM geodesic signal strategy (SR ≈ 1.8) vs buy-and-hold (SR ≈ 1.1)
on Q1 2025 synthetic equity, after 2 bps/side transaction costs.

In [ ]:
import src.figures.fig6_cumulative_pnl as _m6
fig6_out = OUT / 'fig6_cumulative_pnl.png'
_m6.make_figure(fig6_out)
Image(str(fig6_out))

---
## Figure 7 — Covariance Manifold & Geodesic Interpolation

**Paper §5 (Implementation), Figure 7.**  
Three panels: (A) rolling covariance trajectory on the SPD manifold with
geodesic interpolant, (B) geodesic deviation norm ||J(τ)|| over proper time,
(C) Christoffel symbol magnitude heatmap along the trajectory.

In [ ]:
import src.figures.fig7_covariance_manifold as _m7
fig7_out = OUT / 'fig7_covariance_manifold.png'
_m7.make_figure(fig7_out)
Image(str(fig7_out))

---
## Figure 8 — SRFM Module Pipeline

**Paper §5, Figure 8.**  
Data-flow diagram showing each pipeline stage, input/output types,
and measured P50 latency budget from the C++20 implementation.

In [ ]:
import src.figures.fig8_module_pipeline as _m8
fig8_out = OUT / 'fig8_module_pipeline.png'
_m8.make_figure(fig8_out)
Image(str(fig8_out))

---
## Summary — Key Quantitative Results

| Result | Value | Section |
|---|---|---|
| Variance ratio VR (99.5th pctile, n=20) | **1.27×** | §6.2 |
| Bartlett test p-value | **6 × 10⁻¹⁶** | §6.1 |
| TIMELIKE fraction (1-min bars) | ~75% | §6.1 |
| SRFM strategy Sharpe ratio (Q1 2025) | ~1.8 | §6.4 |
| Buy-and-hold Sharpe ratio (Q1 2025) | ~1.1 | §6.4 |
| Pipeline P50 latency (Beta calc) | 48 ns | §5.3 |
| Test-to-production ratio | 1.5:1 | §5.1 |

In [ ]:
# Reproduce the Q1 Bartlett test from scratch using the srfm package
if result is not None:
    from scipy.stats import bartlett, levene
    tl = result.loc[result['spacetime_interval'] < 0, 'relativistic_return'].dropna()
    sl = result.loc[result['spacetime_interval'] > 0, 'relativistic_return'].dropna()
    vr   = sl.var() / tl.var()
    b_s, b_p = bartlett(tl, sl)
    l_s, l_p = levene(tl, sl)
    print(f'Variance ratio (VR):      {vr:.4f}x')
    print(f'Bartlett  stat={b_s:.1f}  p={b_p:.2e}')
    print(f'Levene    stat={l_s:.1f}  p={l_p:.2e}')
    assert vr > 1.0, 'SPACELIKE should have higher variance than TIMELIKE'
    assert b_p < 0.01, 'Variance difference should be statistically significant'
    print('\nAll assertions pass.')
else:
    print('Install srfm-python to run the live validation.')